# 🔍 Bing Grounding Agent — Solution

This notebook provides a complete working solution to the Bing Grounding challenge.
The agent researches Morocco's digital economy using live Bing search.

In [1]:
# ! pip install -r ../../../../requirements.txt -U

In [2]:
from agent_framework.azure import AzureAIAgentClient, AzureAIAgentsProvider
from azure.identity import AzureCliCredential

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    BingGroundingTool,
    BingGroundingSearchToolParameters,
    BingGroundingSearchConfiguration,
)
from azure.core.exceptions import ResourceNotFoundError

In [3]:
import os
import base64
from dotenv import load_dotenv

In [4]:
load_dotenv("../../../../.env")

True

In [5]:
AgentName = "Morocco-BizIntel-Agent"

# SOLUTION — Task 1: Agent Instructions
AgentInstructions = """You are a Morocco Business Intelligence research assistant.
Your role is to help investors and business professionals understand Morocco's
digital economy, tech sector, and business environment.

- Always use the Bing search tool to find current, up-to-date information.
- Cite your sources clearly for every key fact.
- Provide structured, actionable insights.
- At the end of your answer, add one interesting business fact about Casablanca."""

In [6]:
env_connection_name = os.environ["BING_CONNECTION_NAME"]

with AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
) as project:
    bing_connection = None
    try:
        bing_connection = project.connections.get(env_connection_name)
        print(f"Resolved connection: {bing_connection.name} -> {bing_connection.id}")
    except ResourceNotFoundError:
        pass

    if bing_connection is None:
        available = [c.name for c in project.connections.list()]
        raise RuntimeError(
            f"No matching Bing connection found. Tried: {env_connection_name}. "
            f"Available project connections: {available}"
        )

    latest_version = None
    next_version = 1
    try:
        versions = list(project.agents.list_versions(agent_name=AgentName))
        if versions:
            latest_agent = versions[0]
            latest_version = int(latest_agent.version) if hasattr(latest_agent, 'version') else latest_agent.version
            next_version = latest_version + 1
            print(f"Agent '{AgentName}' exists. Creating new version: {next_version}")
    except ResourceNotFoundError:
        print(f"Agent '{AgentName}' does not exist. Creating version: {next_version}")

    model_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME")

    # SOLUTION — Task 2: Configure BingGroundingTool and create agent
    agent = project.agents.create_version(
        agent_name=AgentName,
        definition=PromptAgentDefinition(
            model=model_name,
            instructions=AgentInstructions,
            tools=[
                BingGroundingTool(
                    bing_grounding=BingGroundingSearchToolParameters(
                        search_configurations=[
                            BingGroundingSearchConfiguration(
                                project_connection_id=bing_connection.id
                            )
                        ]
                    )
                )
            ],
        ),
        description="Morocco business intelligence agent with Bing grounding",
    )

    print(f"\n✓ Agent version {agent.version} created successfully")
    print(f"  - Agent ID: {agent.id}")

Resolved connection: hanabingsearchj9kdyn -> /subscriptions/d6233897-5c9f-47f9-8507-6d4ada2d5183/resourceGroups/AgenticAI-AzureMasterclass/providers/Microsoft.CognitiveServices/accounts/hana-foundry-test/projects/proj-default/connections/hanabingsearchj9kdyn

✓ Agent version 1 created successfully
  - Agent ID: Morocco-BizIntel-Agent:1


In [7]:
# SOLUTION — Task 3: A question about Morocco's digital economy
UserQuestion = "What is Morocco's national digital transformation strategy and what progress has been made recently?"

In [8]:
# SOLUTION — Task 4: Query the agent
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=AzureCliCredential(),
)

openai_client = project_client.get_openai_client()

response = openai_client.responses.create(
    input=[{"role": "user", "content": UserQuestion}],
    extra_body={"agent_reference": {"name": AgentName, "version": agent.version, "type": "agent_reference"}},
)

print(f"Response output: {response.output_text}")

Response output: **Morocco’s National Digital Transformation Strategy:**

Morocco’s current national digital transformation blueprint is the “Digital Morocco 2030” strategy. This wide-reaching plan was adopted and updated by Morocco’s Digital Development Agency in late 2025. The strategy is driven by a vision to make Morocco a leading regional digital hub, empowered by technology and innovation, and contributing to economic competitiveness, improved public services, and digital inclusion【4:0†source】【4:2†source】【4:3†source】【4:4†source】.

### Core Pillars and Objectives

**1. Digitalization of Public Services (“e-Gov”):**
- Digitize all essential government services via a unified national portal.
- Generalize the use of shared digital identity and e-signature platforms.
- Target a leap in the UN Online Services Index from 113th to 50th place by 2030.

**2. Development of the Digital Economy:**
- Stimulate digital entrepreneurship, local production of tech solutions, and attract foreign t